In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image

# Config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

IMG_SIZE = 100
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 0.0001
TRAIN_DIR = "Fruits/train/train" # Your labeled data source

Using device: cuda


In [16]:
# 1. Define Transforms
# Heavy augmentation for training to prevent overfitting
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Clean transform for validation (just resize and normalize)
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# 2. Load the data
# We load the entire labeled dataset
full_dataset = datasets.ImageFolder(root=TRAIN_DIR)
class_names = full_dataset.classes
print(f"Found {len(class_names)} classes.")

# 3. Create the Split (80% Train, 20% Validation)
torch.manual_seed(42) # For reproducibility
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

# Generate random indices
indices = torch.randperm(len(full_dataset)).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

# 4. Create Subsets with correct transforms
# We use a wrapper to apply specific transforms to the subsets
class SubsetWithTransform(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
        
    def __len__(self):
        return len(self.subset)

# Create raw subsets first
raw_train_subset = Subset(full_dataset, train_indices)
raw_val_subset = Subset(full_dataset, val_indices)

# Apply transforms
train_dataset = SubsetWithTransform(raw_train_subset, transform=train_transform)
val_dataset = SubsetWithTransform(raw_val_subset, transform=val_transform)

# 5. DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training on {len(train_dataset)} images")
print(f"Validating on {len(val_dataset)} images")

Found 33 classes.
Training on 13483 images
Validating on 3371 images


In [17]:
class OptimizedFruitCNN(nn.Module):
    def __init__(self, num_classes):
        super(OptimizedFruitCNN, self).__init__()
        
        # Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        
        # Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Classifier
        self.flatten = nn.Flatten()
        # 100x100 -> 12x12 output after 3 pools
        self.fc1 = nn.Linear(128 * 12 * 12, 512)
        self.dropout = nn.Dropout(0.5) 
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = OptimizedFruitCNN(len(class_names)).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_loss = float('inf')

print("Starting Training...")

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    avg_train_loss = running_loss / len(train_loader)
    avg_train_acc = correct / total
    
    # Validation
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    avg_val_loss = val_loss / len(val_loader)
    avg_val_acc = correct / total
    
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    train_accs.append(avg_train_acc)
    val_accs.append(avg_val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} Acc: {avg_train_acc:.4f} | Val Loss: {avg_val_loss:.4f} Acc: {avg_val_acc:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')

print("Done!")

Starting Training...


In [ ]:
# Predict on the unlabeled test folder
TEST_DIR = "Fruits/test/test" # Adjust based on your 'test/test/image.png' structure

if os.path.exists(TEST_DIR):
    model.load_state_dict(torch.load('best_model.pth'))
    model.eval()
    
    image_files = os.listdir(TEST_DIR)[:5] # Show first 5
    
    plt.figure(figsize=(15, 3))
    for i, img_file in enumerate(image_files):
        img_path = os.path.join(TEST_DIR, img_file)
        img = Image.open(img_path).convert('RGB')
        
        # Transform
        img_tensor = val_transform(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(img_tensor)
            _, pred = torch.max(output, 1)
            
        predicted_class = class_names[pred.item()]
        
        plt.subplot(1, 5, i+1)
        plt.imshow(img)
        plt.title(predicted_class)
        plt.axis('off')
    plt.show()
else:
    print(f"Could not find test folder at {TEST_DIR}")